## Understanding ETA behavior
In this stage we'll try and understand how reported AIS ETA behaves over time for each vessel before we build anything on top of it. First step is to load the cleaned dataset in the new notebook

In [1]:
import pandas as pd

df_target = pd.read_csv('../outputs/ais_cleaned.csv')


Its really important that we understand ETA volatility across the dataset for any future calculations

In [ ]:
pd.set_option('Display.float_format', lambda x: f"{x:.2f}")
eta_count = df_target.groupby('MMSI')['ETA'].nunique()
single_eta = eta_count[eta_count == 1]
mult_eta = eta_count[eta_count > 1]
print(f"{((mult_eta.count() / eta_count.count()) * 100):.2f}%")
print(eta_count.describe())

Across 867 unique vessels the mean number of unique ETA values is 1.35 while 23.76% of vessels have more than 1 unique ETA value. These numbers suggest that reported ETA is fairly stable over the 24 hour window. The next important question is: for the vessels with multiple ETA values, what do those changes look over time? Are they expected gradual changes or are they big time jumps?

In [ ]:
print(mult_eta[mult_eta == 6])
eta_6 = df_target[df_target['MMSI'] == 256672000 ]
eta_6 = eta_6.sort_values('# Timestamp', ascending=True)
eta_6 = eta_6[eta_6['ETA'] != eta_6['ETA'].shift()]
print(eta_6[['# Timestamp', 'ETA', 'Destination', 'SOG']])

One of the three vessels with 6 unique reported ETA values appears to complete its Gothenburg voyage leg -> update its destination to PLGDY while basically stationary at 0.30kn and subsequently depart on the next voyage leg. This shows us that our original filter(number of unique ETA values per MMSI) possibly mixes multiple voyage legs/voyages with completely different ETA values in many vessels meaning it doesnt measure true ETA volatility. This tells us that in order to maintain accuracy in our analysis we need to ultimately measure ETA changes within the same destination/voyage leg rather than treat all ETA changes as one continuous voyage.

In [ ]:
corr_eta_count = df_target.groupby(['MMSI', 'Destination'])['ETA'].nunique()
corr_single_eta = corr_eta_count[corr_eta_count == 1]
corr_mult_eta = corr_eta_count[corr_eta_count > 1]
print(f"{((corr_mult_eta.count() / corr_eta_count.count()) * 100):.2f}%")
print(corr_eta_count.describe())

Once destination changes are accounted for, most vessel-destination groups report a single ETA during the 24 hour observation window. Only about 17.84% of vessel-destination groups report more than one ETA suggesting that reported ETA volatility is generally stable within each voyage/voyage leg. !important distinction: because each destination/port still has many aliases the same destination may get split into two different ones reducing accuracy of the numbers above.

In [ ]:
df_target['ETA'] = pd.to_datetime(df_target['ETA'])
df_target['prev_eta'] = df_target.groupby(['MMSI', 'Destination'])['ETA'].shift(1)
df_target['time_delta'] = df_target['ETA'] - df_target['prev_eta']
eta_vol = df_target[(df_target['time_delta'] != pd.Timedelta(0)) & (df_target['time_delta'].notnull())].copy()
eta_vol['time_delta_hours'] = eta_vol['time_delta'].dt.total_seconds() / 3600
print(eta_vol['time_delta_hours'].describe())
eta_vol['abs_change_hours'] = eta_vol['time_delta_hours'].abs()
print(eta_vol['abs_change_hours'].sort_values(ascending=False).head(30))

We can see that the majority of time-delta values are completely normal with 75% of values being less than 10.58 hours. We have also caught some extreme outliers like -8602 and 8602 hours. Lets investigate

In [ ]:
print(eta_vol[eta_vol['time_delta_hours'] == -8602])
print(eta_vol[eta_vol['time_delta_hours'] == 8602])
ext_count = eta_vol['abs_change_hours'][eta_vol['abs_change_hours'] >= 720].count()
print(f"{((ext_count / (eta_vol['abs_change_hours'].count())) * 100):.2f}%")

The same vessel called ROBIN HOOD is responsible for both the max and min time-delta values and if we look at the reported ETA it jumps forward to 2025 more then likely hinting to a malformed ETA. The extreme time-deltas (using an extreme absolute threshold of 720 hours) account for 7.09% of all time deltas with a maximum value of 8602 hours and a minimum value of 720 hours. Theres also an odd amount of 720 hour values. We'll inspect these before we ultimately decide if 720 hours is the extreme threshold

In [ ]:
print(eta_vol[eta_vol['abs_change_hours'] == 720])

After further investigating the suspicious values of 720 hours we once again find ourselves dealing with a vessel that reports the same eta 30 days apart back and forth hinting at another malformed eta. We can now confidently set the extreme absolute threshold to 720 hours 

In [ ]:
eta_vol = eta_vol[eta_vol['abs_change_hours'] < 720]
print(eta_vol['abs_change_hours'].describe())

After filtering out malformed outliers we can see ETA volatility is relatively modest with a mean absolute time-delta of 7.6 hours while 75% of ETA values are below 10.69 hours. Finally we will be filtering the dataset itself. We wont use the same criteria we used to measure actual volatility since it would remove the malformed row aswell as the next row since the timedelta from the malformed row to the next one will also be above the absolute threshold. Instead we will use the indices of the exact malformed rows and selectively remove them from the dataset

In [ ]:
extreme_index = [3841061, 3838692, 4348483, 8789155, 8446443, 8589232, 3096335, 8789988, 7518033, 7791647, 4174405, 7402325, 3160393, 1879665, 7770367, 8481932, 8372223, 7270780]
print(df_target.shape[0])
df_target = df_target[~(df_target.index.isin(extreme_index))].copy()
df_target['prev_eta'] = df_target.groupby(['MMSI', 'Destination'])['ETA'].shift(1)
df_target['time_delta'] = df_target['ETA'] - df_target['prev_eta']
print(df_target.shape[0])

We've gone from 9,751,600 rows down to 9,751,582 rows

In [30]:
df_target = df_target.drop(columns=['prev_eta', 'time_delta'])
df_target.to_csv('../outputs/ais_eta_validated.csv', index=False)